In [1]:
!pip install -q \
    pymupdf \
    langchain \
    langchain-community \
    langchain-groq \
    sentence-transformers \
    faiss-cpu \
    gradio \
    mlflow \
    bitsandbytes \
    accelerate \
    transformers \
    rank_bm25 \
    reportlab

print("All dependencies installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.0 MB/s eta 0:00:00
All dependencies installed.


In [ ]:
import os, re
import fitz
from pathlib import Path
from dataclasses import dataclass
from typing import List
from collections import Counter

from google.colab import userdata
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

os.environ["GROQ_API_KEY"]    = userdata.get("GROQ_API_KEY")
os.environ["GITHUB_TOKEN"]    = userdata.get("GITHUB_TOKEN")
os.environ["NGROK_AUTHTOKEN"] = userdata.get("NGROK_AUTHTOKEN")

print("Secrets loaded.")

In [3]:
@dataclass
class RAGConfig:
    chunk_size: int        = 512
    chunk_overlap: int     = 64
    min_chunk_length: int  = 50
    top_k_retrieval: int   = 10
    top_k_rerank: int      = 4
    llm_model: str         = "llama-3.1-8b-instant"
    temperature: float     = 0.2
    max_output_tokens: int = 1024

    embed_model: str       = "sentence-transformers/all-MiniLM-L6-v2"

    vector_store_dir: str  = "/content/vector_store"
    upload_dir: str        = "/content/uploaded_pdfs"


cfg = RAGConfig()
print(f"Config  |  chunk_size={cfg.chunk_size}  overlap={cfg.chunk_overlap}  "
      f"top_k={cfg.top_k_retrieval}  rerank_k={cfg.top_k_rerank}")

Config  |  chunk_size=512  overlap=64  top_k=10  rerank_k=4


In [4]:
class PDFIngestionPipeline:

    _NOISE_PATTERNS = [
        r"^\s*\d+\s*$",
        r"^confidential\s*$",
        r"^all rights reserved",
    ]

    def __init__(self, config: RAGConfig):
        self.cfg = config

    def ingest(self, pdf_paths: List[str]) -> List[Document]:
        all_docs = []
        for path in pdf_paths:
            docs = self._process_file(path)
            all_docs.extend(docs)
            print(f"  {Path(path).name}  →  {len(docs)} pages extracted")
        print(f"\nTotal pages: {len(all_docs)}")
        return all_docs

    def _process_file(self, pdf_path: str) -> List[Document]:
        docs = []
        pdf  = fitz.open(pdf_path)
        total = len(pdf)
        for page_num, page in enumerate(pdf, start=1):
            text = self._clean_text(page.get_text("text"))
            if len(text) < self.cfg.min_chunk_length:
                continue
            docs.append(Document(
                page_content=text,
                metadata={
                    "source":      Path(pdf_path).name,
                    "page":        page_num,
                    "total_pages": total,
                    "file_path":   pdf_path,
                }
            ))
        pdf.close()
        return docs

    def _clean_text(self, text: str) -> str:
        text  = re.sub(r"\n{3,}", "\n\n", text)
        text  = re.sub(r"[ \t]+",  " ",    text)
        lines = [
            l for l in text.splitlines()
            if not any(re.match(p, l.strip().lower()) for p in self._NOISE_PATTERNS)
        ]
        return "\n".join(lines).strip()


print("PDFIngestionPipeline defined.")

PDFIngestionPipeline defined.


In [5]:
class ChunkingPipeline:

    def __init__(self, config: RAGConfig):
        self.cfg = config
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size    = config.chunk_size,
            chunk_overlap = config.chunk_overlap,
            separators    = ["\n\n", "\n", ". ", " ", ""],
            length_function = len,
        )

    def split(self, documents: List[Document]) -> List[Document]:
        chunks = []
        for doc in documents:
            for idx, chunk in enumerate(self.splitter.split_documents([doc])):
                if len(chunk.page_content.strip()) < self.cfg.min_chunk_length:
                    continue
                chunk.metadata["chunk_index"] = idx
                chunk.metadata["chunk_id"]    = (
                    f"{chunk.metadata['source']}_p{chunk.metadata['page']}_c{idx}"
                )
                chunks.append(chunk)
        print(f"Total chunks produced: {len(chunks)}")
        return chunks


print("ChunkingPipeline defined.")

ChunkingPipeline defined.


In [6]:
def inspect_chunks(chunks: List[Document], sample_n: int = 3) -> None:
    lengths = [len(c.page_content) for c in chunks]
    sources = Counter(c.metadata["source"] for c in chunks)

    print("=" * 60)
    print("CHUNK STATISTICS")
    print("=" * 60)
    print(f"  Total    : {len(chunks)}")
    print(f"  Min len  : {min(lengths)} chars")
    print(f"  Max len  : {max(lengths)} chars")
    print(f"  Avg len  : {sum(lengths)/len(lengths):.0f} chars")
    print(f"  Median   : {sorted(lengths)[len(lengths)//2]} chars")
    print()
    print("  Chunks per file:")
    for src, n in sources.items():
        print(f"    {src}: {n}")
    print()
    print(f"  Samples (first {sample_n}):")
    print("-" * 60)
    for chunk in chunks[:sample_n]:
        print(f"  ID: {chunk.metadata['chunk_id']}")
        print(f"  {chunk.page_content[:200].replace(chr(10), ' ')}...")
        print()


print("inspect_chunks() defined.")

inspect_chunks() defined.


In [7]:
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.units import inch

SAMPLE_PDF_PATH = "/content/sample_rag_test.pdf"

SAMPLE_CONTENT = [
    ("Introduction to Retrieval-Augmented Generation",
     """Retrieval-Augmented Generation (RAG) is a hybrid AI architecture that combines the
parametric knowledge of large language models with non-parametric, external knowledge sources.
Rather than relying solely on information encoded in model weights during training, RAG systems
dynamically retrieve relevant documents at inference time and condition the generation on that
retrieved context. This approach addresses key limitations of standalone LLMs: knowledge
staleness, hallucination, and the inability to cite sources."""),

    ("Vector Databases and Semantic Search",
     """At the core of any RAG pipeline lies a vector database. Documents are split into chunks,
each chunk is encoded into a dense embedding vector using a transformer encoder model such as
sentence-transformers. These vectors capture semantic meaning: two chunks about similar topics
will have embeddings close together in high-dimensional space. FAISS (Facebook AI Similarity
Search) is a widely used open-source library for efficient approximate nearest-neighbour search
over millions of vectors. It supports several index types: Flat (exact, slow), IVF (inverted
file, faster), and HNSW (graph-based, fastest for large corpora)."""),

    ("Chunking Strategies",
     """How a document is split into chunks significantly impacts retrieval quality. Fixed-size
character splitting is simple but cuts across sentence and paragraph boundaries. Recursive
character splitting uses a priority list of separators to find natural split points. Semantic
chunking groups sentences by embedding similarity rather than character count, producing
topically coherent chunks at the cost of higher preprocessing time. Chunk size is a key
hyperparameter: larger chunks give more context per retrieval but reduce precision; smaller
chunks are more precise but may lack context for generation."""),

    ("Re-ranking and Maximal Marginal Relevance",
     """First-stage retrieval with dense embeddings optimises for relevance but can return
redundant chunks that all say the same thing. Maximal Marginal Relevance (MMR) selects chunks
iteratively: each selected chunk must be relevant to the query but dissimilar to already-selected
chunks. Cross-encoder re-ranking uses a separate model that takes a query-chunk pair as input
and outputs a fine-grained relevance score. Cross-encoders are more accurate than bi-encoders
for ranking but cannot be used for first-stage retrieval."""),

    ("Evaluation Metrics for RAG Systems",
     """Evaluating RAG systems requires measuring both retrieval quality and generation quality.
For retrieval: Recall@K measures what fraction of ground-truth relevant documents appear in
the top K results; MRR rewards systems that rank the correct document higher. For generation:
RAGAS provides reference-free metrics including faithfulness, answer relevancy, and context
precision. Latency is also critical: end-to-end query latency should stay under 3 seconds
for interactive applications."""),
]

doc = SimpleDocTemplate(SAMPLE_PDF_PATH, pagesize=letter,
                        topMargin=inch, bottomMargin=inch)
styles = getSampleStyleSheet()
story  = []
for title, body in SAMPLE_CONTENT:
    story.append(Paragraph(title, styles["Heading2"]))
    story.append(Spacer(1, 0.1 * inch))
    story.append(Paragraph(body, styles["BodyText"]))
    story.append(Spacer(1, 0.3 * inch))
doc.build(story)

print(f"Sample PDF created: {SAMPLE_PDF_PATH}")
pdf_paths = [SAMPLE_PDF_PATH]

Sample PDF created: /content/sample_rag_test.pdf


In [8]:
from google.colab import files

os.makedirs(cfg.upload_dir, exist_ok=True)
uploaded = files.upload()

pdf_paths = []
for fname, content in uploaded.items():
    dest = os.path.join(cfg.upload_dir, fname)
    with open(dest, "wb") as f:
        f.write(content)
    pdf_paths.append(dest)
    print(f"Saved: {dest}")

Saving sample_rag_test.pdf to sample_rag_test (3).pdf
Saved: /content/uploaded_pdfs/sample_rag_test (3).pdf


In [9]:
ingestion_pipeline = PDFIngestionPipeline(cfg)
pages = ingestion_pipeline.ingest(pdf_paths)

chunking_pipeline = ChunkingPipeline(cfg)
chunks = chunking_pipeline.split(pages)

inspect_chunks(chunks, sample_n=3)



  sample_rag_test (3).pdf  →  1 pages extracted

Total pages: 1
Total chunks produced: 6
CHUNK STATISTICS
  Total    : 6
  Min len  : 197 chars
  Max len  : 480 chars
  Avg len  : 407 chars
  Median   : 442 chars

  Chunks per file:
    sample_rag_test (3).pdf: 6

  Samples (first 3):
------------------------------------------------------------
  ID: sample_rag_test (3).pdf_p1_c0
  Introduction to Retrieval-Augmented Generation Retrieval-Augmented Generation (RAG) is a hybrid AI architecture that combines the parametric knowledge of large language models with non-parametric exte...

  ID: sample_rag_test (3).pdf_p1_c1
  This approach addresses key limitations of standalone LLMs: knowledge staleness, hallucination, and the inability to cite sources. Vector Databases and Semantic Search At the core of any RAG pipeline ...

  ID: sample_rag_test (3).pdf_p1_c2
  efficient approximate nearest-neighbour search over millions of vectors. It supports several index types: Flat for exact search, 

In [10]:
!pip install -q langchain-huggingface

from langchain_huggingface import HuggingFaceEmbeddings

print("Loading embedding model... (downloads ~80MB on first run)")

embeddings = HuggingFaceEmbeddings(
    model_name=cfg.embed_model,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

test_vec = embeddings.embed_query("test sentence")
print(f"Model loaded. Embedding dimension: {len(test_vec)}")
print("Expected: 384")

Loading embedding model... (downloads ~80MB on first run)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded. Embedding dimension: 384
Expected: 384


In [11]:
import time
from langchain_community.vectorstores import FAISS

os.makedirs(cfg.vector_store_dir, exist_ok=True)

print(f"Embedding {len(chunks)} chunks...")
t0 = time.time()

vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings,
)

elapsed = time.time() - t0
print(f"Done. {len(chunks)} chunks embedded in {elapsed:.1f}s")
print(f"FAISS index size: {vector_store.index.ntotal} vectors")

vector_store.save_local(cfg.vector_store_dir)
print(f"Vector store saved to: {cfg.vector_store_dir}")

/tmp/ipykernel_10098/3652974368.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Embedding 6 chunks...
Done. 6 chunks embedded in 0.3s
FAISS index size: 6 vectors
Vector store saved to: /content/vector_store


In [12]:
vector_store = FAISS.load_local(
    cfg.vector_store_dir,
    embeddings,
    allow_dangerous_deserialization=True,
)
print(f"Reloaded index. Vectors in store: {vector_store.index.ntotal}")

test_query = "What is MMR and why is it used in retrieval?"
print(f"\nTest query: '{test_query}'")

print("\n--- Standard Similarity Search (top 3) ---")
sim_results = vector_store.similarity_search_with_score(test_query, k=3)
for i, (doc, score) in enumerate(sim_results, 1):
    print(f"\n[{i}] Score (L2 distance, lower=better): {score:.4f}")
    print(f"    Source: {doc.metadata['chunk_id']}")
    print(f"    Text:   {doc.page_content[:150].replace(chr(10), ' ')}...")

print("\n--- MMR Retrieval (fetch 6, keep 3, lambda=0.7) ---")
mmr_results = vector_store.max_marginal_relevance_search(
    test_query,
    k=cfg.top_k_rerank,
    fetch_k=cfg.top_k_retrieval,
    lambda_mult=0.7,
)
for i, doc in enumerate(mmr_results, 1):
    print(f"\n[{i}] Source: {doc.metadata['chunk_id']}")
    print(f"    Text:   {doc.page_content[:150].replace(chr(10), ' ')}...")


Reloaded index. Vectors in store: 6

Test query: 'What is MMR and why is it used in retrieval?'

--- Standard Similarity Search (top 3) ---

[1] Score (L2 distance, lower=better): 1.1209
    Source: sample_rag_test (3).pdf_p1_c4
    Text:   already-selected chunks. Cross-encoder re-ranking scores each query-chunk pair with higher accuracy than bi-encoders, at the cost of not being usable ...

[2] Score (L2 distance, lower=better): 1.2932
    Source: sample_rag_test (3).pdf_p1_c3
    Text:   priority list of separators to find natural split points. Chunk size is a critical hyperparameter: larger chunks give more context per retrieval but r...

[3] Score (L2 distance, lower=better): 1.4518
    Source: sample_rag_test (3).pdf_p1_c1
    Text:   This approach addresses key limitations of standalone LLMs: knowledge staleness, hallucination, and the inability to cite sources. Vector Databases an...

--- MMR Retrieval (fetch 6, keep 3, lambda=0.7) ---

[1] Source: sample_rag_test (3).pdf_p1_c4

In [13]:
!pip install -q sentence-transformers

from sentence_transformers import CrossEncoder

print("Loading cross-encoder re-ranker...")
reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2",
    max_length=512,
    device="cpu",
)

test_scores = reranker.predict([
    ("What is MMR?", "MMR selects diverse chunks relevant to the query."),
    ("What is MMR?", "The capital of France is Paris."),
])
print(f"Re-ranker loaded.")
print(f"Relevance score (good pair): {test_scores[0]:.4f}")
print(f"Relevance score (bad pair) : {test_scores[1]:.4f}")
print("Expected: good pair score >> bad pair score")

Loading cross-encoder re-ranker...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Re-ranker loaded.
Relevance score (good pair): 5.4910
Relevance score (bad pair) : -11.2699
Expected: good pair score >> bad pair score


In [14]:
from langchain_core.retrievers import BaseRetriever
from langchain_core.documents import Document
from pydantic import Field
from typing import List
import numpy as np

class ReRankingRetriever(BaseRetriever):
    """
    MMR retrieval followed by cross-encoder re-ranking.
    Plug-and-play: swap vector_store or reranker without changing the chain.
    """

    vector_store: object = Field(description="FAISS vector store instance")
    reranker:     object = Field(description="CrossEncoder model instance")
    top_k_fetch:  int    = Field(default=10)
    top_k_final:  int    = Field(default=4)
    lambda_mult:  float  = Field(default=0.7)

    class Config:
        arbitrary_types_allowed = True

    def _get_relevant_documents(self, query: str) -> List[Document]:
        candidates = self.vector_store.max_marginal_relevance_search(
            query,
            k=self.top_k_fetch,
            fetch_k=self.top_k_fetch * 2,
            lambda_mult=self.lambda_mult,
        )

        if not candidates:
            return []

        pairs  = [(query, doc.page_content) for doc in candidates]
        scores = self.reranker.predict(pairs)

        ranked = sorted(
            zip(scores, candidates),
            key=lambda x: x[0],
            reverse=True,
        )[:self.top_k_final]

        results = []
        for score, doc in ranked:
            doc.metadata["rerank_score"] = float(score)
            results.append(doc)

        return results

    async def _aget_relevant_documents(self, query: str) -> List[Document]:
        return self._get_relevant_documents(query)


retriever = ReRankingRetriever(
    vector_store=vector_store,
    reranker=reranker,
    top_k_fetch=cfg.top_k_retrieval,
    top_k_final=cfg.top_k_rerank,
    lambda_mult=0.7,
)

print("ReRankingRetriever ready.")

ReRankingRetriever ready.


/tmp/ipykernel_10098/1773531235.py:7: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  class ReRankingRetriever(BaseRetriever):


In [15]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

cfg.llm_model = "llama-3.1-8b-instant"

llm = ChatGroq(
    model=cfg.llm_model,
    api_key=os.environ["GROQ_API_KEY"],
    temperature=cfg.temperature,
    max_tokens=cfg.max_output_tokens,
)

RAG_PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template="""You are a precise document assistant. Answer the question using ONLY
the context provided below. Do not use any outside knowledge.
If the context does not contain enough information to answer, say:
"I don't have enough information in the provided documents to answer this."

Context:
{context}

Question: {question}

Answer (be concise and cite which section your answer comes from):"""
)

def format_docs(docs):
    return "\n\n".join(
        f"[{doc.metadata['chunk_id']}]\n{doc.page_content}" for doc in docs
    )

rag_chain = (
    {
        "context":  retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

print(f"RAG chain ready.")
print(f"LLM      : {cfg.llm_model}")
print(f"Retrieval: MMR (fetch {cfg.top_k_retrieval}) → re-rank → top {cfg.top_k_rerank}")

RAG chain ready.
LLM      : llama-3.1-8b-instant
Retrieval: MMR (fetch 10) → re-rank → top 4


In [16]:
import time

def query_rag(question: str, verbose: bool = True) -> dict:
    """
    Runs the full RAG pipeline and returns answer + sources + latency.
    Reused by Gradio (Step 5) and MLflow (Step 6).
    """
    t0 = time.time()

    answer  = rag_chain.invoke(question)
    sources = retriever.invoke(question)

    latency = time.time() - t0

    if verbose:
        print(f"Question : {question}")
        print(f"Latency  : {latency:.2f}s")
        print(f"\nAnswer:\n{answer}")
        print(f"\nSources used ({len(sources)}):")
        for i, doc in enumerate(sources, 1):
            score = doc.metadata.get("rerank_score", "N/A")
            score_str = f"{score:.4f}" if isinstance(score, float) else score
            print(f"  [{i}] {doc.metadata['chunk_id']}  |  rerank_score={score_str}")

    return {"answer": answer, "sources": sources, "latency": latency}


print("=" * 60)
result1 = query_rag("What is MMR and why is it used in retrieval?")
print("\n" + "=" * 60)
result2 = query_rag("What evaluation metrics are used for RAG systems?")
print("\n" + "=" * 60)
result3 = query_rag("What is the capital of Japan?")

Question : What is MMR and why is it used in retrieval?
Latency  : 3.19s

Answer:
Maximal Marginal Relevance (MMR) is used in retrieval to select chunks that are relevant to the query but dissimilar to previously selected chunks. This is to avoid returning redundant chunks. (sample_rag_test (3).pdf_p1_c3)

Sources used (4):
  [1] sample_rag_test (3).pdf_p1_c0  |  rerank_score=-1.0971
  [2] sample_rag_test (3).pdf_p1_c2  |  rerank_score=-6.0367
  [3] sample_rag_test (3).pdf_p1_c4  |  rerank_score=-6.3722
  [4] sample_rag_test (3).pdf_p1_c3  |  rerank_score=-6.9984

Question : What evaluation metrics are used for RAG systems?
Latency  : 2.36s

Answer:
Recall at K and reference-free metrics including faithfulness, answer relevancy, and context precision (from [sample_rag_test (3).pdf_p1_c0] and [sample_rag_test (3).pdf_p1_c5]).

Sources used (4):
  [1] sample_rag_test (3).pdf_p1_c4  |  rerank_score=7.8478
  [2] sample_rag_test (3).pdf_p1_c0  |  rerank_score=0.8258
  [3] sample_rag_test (3

In [17]:
import gradio as gr
import time

def chat_with_docs(question, history):
    if not question.strip():
        return history, "", ""

    result = query_rag(question, verbose=False)

    answer  = result["answer"]
    sources = result["sources"]
    latency = result["latency"]

    sources_md = f"**Query latency: {latency:.2f}s**\n\n---\n\n**Sources used ({len(sources)}):**\n\n"
    for i, doc in enumerate(sources, 1):
        score     = doc.metadata.get("rerank_score", "N/A")
        score_str = f"{score:.4f}" if isinstance(score, float) else score
        chunk_id  = doc.metadata["chunk_id"]
        preview   = doc.page_content[:200].replace("\n", " ")
        sources_md += f"**[{i}] {chunk_id}** | score: `{score_str}`\n\n> {preview}...\n\n"

    history.append((question, answer))
    return history, "", sources_md


def clear_all():
    return [], "", ""


with gr.Blocks(title="RAG Document Chatbot") as demo:

    gr.Markdown("## RAG Document Intelligence Chatbot")
    gr.Markdown("Ask questions about your uploaded PDF. The system retrieves relevant chunks using MMR + cross-encoder re-ranking.")

    with gr.Row():
        with gr.Column(scale=2):
            chatbot = gr.Chatbot(
                label="Conversation",
                height=450,
                show_copy_button=True,
            )
            with gr.Row():
                question_box = gr.Textbox(
                    placeholder="Ask a question about your document...",
                    label="Your Question",
                    scale=4,
                    lines=1,
                )
                submit_btn = gr.Button("Ask", variant="primary", scale=1)

            clear_btn = gr.Button("Clear conversation", variant="secondary")

        with gr.Column(scale=1):
            sources_panel = gr.Markdown(
                value="*Sources will appear here after your first question.*",
                label="Retrieved Sources",
            )

    gr.Examples(
        examples=[
            ["What is MMR and why is it used in retrieval?"],
            ["What evaluation metrics are used for RAG systems?"],
            ["Explain the difference between FAISS index types."],
            ["What is the capital of Japan?"],
        ],
        inputs=question_box,
        label="Try these example questions",
    )

    submit_btn.click(
        fn=chat_with_docs,
        inputs=[question_box, chatbot],
        outputs=[chatbot, question_box, sources_panel],
    )
    question_box.submit(
        fn=chat_with_docs,
        inputs=[question_box, chatbot],
        outputs=[chatbot, question_box, sources_panel],
    )
    clear_btn.click(
        fn=clear_all,
        outputs=[chatbot, question_box, sources_panel],
    )

demo.launch(share=True, debug=False)

/tmp/ipykernel_10098/2139921272.py:37: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(
/tmp/ipykernel_10098/2139921272.py:37: DeprecationWarning: The 'show_copy_button' parameter will be removed in Gradio 6.0. You will need to use 'buttons=["copy"]' instead.
  chatbot = gr.Chatbot(
/tmp/ipykernel_10098/2139921272.py:37: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fd740d9387b5d3419a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [18]:
import warnings
warnings.filterwarnings("ignore")

In [22]:
import mlflow
import mlflow.sklearn
from datetime import datetime

EXPERIMENT_NAME = "rag_document_chatbot"
mlflow.set_experiment(EXPERIMENT_NAME)

print(f"MLflow tracking URI : {mlflow.get_tracking_uri()}")
print(f"Experiment          : {EXPERIMENT_NAME}")


def query_rag_logged(question: str, verbose: bool = True) -> dict:
    with mlflow.start_run(run_name=f"query_{datetime.now().strftime('%H%M%S')}"):

        mlflow.log_params({
            "llm_model"      : cfg.llm_model,
            "embed_model"    : cfg.embed_model,
            "chunk_size"     : cfg.chunk_size,
            "chunk_overlap"  : cfg.chunk_overlap,
            "top_k_retrieval": cfg.top_k_retrieval,
            "top_k_rerank"   : cfg.top_k_rerank,
            "temperature"    : cfg.temperature,
        })

        mlflow.set_tag("question", question[:250])

        result  = query_rag(question, verbose=verbose)
        answer  = result["answer"]
        sources = result["sources"]
        latency = result["latency"]

        rerank_scores = [
            doc.metadata.get("rerank_score", 0.0)
            for doc in sources
            if isinstance(doc.metadata.get("rerank_score"), float)
        ]

        mlflow.log_metrics({
            "latency_seconds"   : round(latency, 4),
            "sources_retrieved" : len(sources),
            "answer_length"     : len(answer),
            "top_rerank_score"  : round(max(rerank_scores), 4) if rerank_scores else 0.0,
            "mean_rerank_score" : round(sum(rerank_scores) / len(rerank_scores), 4) if rerank_scores else 0.0,
            "min_rerank_score"  : round(min(rerank_scores), 4) if rerank_scores else 0.0,
        })

        artifact_lines = [
            f"Question : {question}",
            f"Latency  : {latency:.2f}s",
            f"Answer   :\n{answer}",
            "\nSources:",
        ]
        for i, doc in enumerate(sources, 1):
            score = doc.metadata.get("rerank_score", "N/A")
            score_str = f"{score:.4f}" if isinstance(score, float) else score
            artifact_lines.append(
                f"  [{i}] {doc.metadata['chunk_id']} | rerank_score={score_str}"
            )

        artifact_path = f"/tmp/query_result_{datetime.now().strftime('%H%M%S')}.txt"
        with open(artifact_path, "w") as f:
            f.write("\n".join(artifact_lines))
        mlflow.log_artifact(artifact_path)

    return result


print("\n" + "=" * 60)
print("Running logged queries...")
print("=" * 60)

logged_questions = [
    "What is MMR and why is it used in retrieval?",
    "What evaluation metrics are used for RAG systems?",
    "Explain the difference between FAISS index types.",
    "What is the capital of Japan?",
    "How does chunking strategy affect retrieval quality?",
]

for q in logged_questions:
    print(f"\nLogging: {q}")
    query_rag_logged(q, verbose=False)

print("\n" + "=" * 60)
print("All queries logged to MLflow.")


client = mlflow.tracking.MlflowClient()
exp    = client.get_experiment_by_name(EXPERIMENT_NAME)
runs   = client.search_runs(
    experiment_ids=[exp.experiment_id],
    order_by=["metrics.latency_seconds ASC"],
)

print(f"\n{'Question':<45} {'Latency':>8}  {'Top Score':>10}  {'Ans Len':>8}")
print("-" * 80)
for run in runs:
    q       = run.data.tags.get("question", "N/A")[:44]
    latency = run.data.metrics.get("latency_seconds", 0)
    top_s   = run.data.metrics.get("top_rerank_score", 0)
    ans_len = run.data.metrics.get("answer_length", 0)
    print(f"{q:<45} {latency:>7.2f}s  {top_s:>10.4f}  {ans_len:>8}")


import pandas as pd

rows = []
for run in runs:
    rows.append({
        "question"        : run.data.tags.get("question", "N/A"),
        "latency_seconds" : run.data.metrics.get("latency_seconds"),
        "top_rerank_score": run.data.metrics.get("top_rerank_score"),
        "mean_rerank_score": run.data.metrics.get("mean_rerank_score"),
        "answer_length"   : run.data.metrics.get("answer_length"),
        "llm_model"       : run.data.params.get("llm_model"),
        "chunk_size"      : run.data.params.get("chunk_size"),
    })

df = pd.DataFrame(rows)
csv_path = "/content/mlflow_runs.csv"
df.to_csv(csv_path, index=False)

print(f"\nRuns exported to: {csv_path}")
print("\nDataFrame preview:")
print(df.to_string(index=False))

print(f"\nMLflow DB saved at: {mlflow.get_tracking_uri()}")

MLflow tracking URI : sqlite:////content/mlflow.db
Experiment          : rag_document_chatbot

Running logged queries...

Logging: What is MMR and why is it used in retrieval?

Logging: What evaluation metrics are used for RAG systems?

Logging: Explain the difference between FAISS index types.

Logging: What is the capital of Japan?

Logging: How does chunking strategy affect retrieval quality?

All queries logged to MLflow.

Question                                       Latency   Top Score   Ans Len
--------------------------------------------------------------------------------
What is the capital of Japan?                    0.59s    -11.0624      73.0
How does chunking strategy affect retrieval      0.64s      7.2078     354.0
What is MMR and why is it used in retrieval?     0.69s     -1.0971     254.0
What is MMR and why is it used in retrieval?     0.69s     -1.0971     275.0
How does chunking strategy affect retrieval      0.76s      7.2078     409.0
What evaluation metrics ar